In [1]:
import json
from collections import defaultdict

import gc_utils
import h5py
import numpy as np
import pandas as pd
import scipy
from sklearn import cluster as cl
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

| Metric        | Meaning                                                                   |
| ------------- | ------------------------------------------------------------------------- |
| **Precision** | Of the predicted class instances, how many were correct? (TP / (TP + FP)) |
| **Recall**    | Of the actual class instances, how many were retrieved? (TP / (TP + FN))  |
| **F1-score**  | Harmonic mean of precision and recall. Good overall performance measure.  |
| **Support**   | Number of true instances for each class.                                  |


In [2]:
def clustering(X, rnd_grps, n_clusters, cluster_type):
    scaler = StandardScaler()
    X_scal = scaler.fit_transform(X)

    if cluster_type == "kmeans":
        kmeans = cl.KMeans(n_clusters=n_clusters, random_state=0).fit(X_scal)
        labels = kmeans.labels_

    if cluster_type == "agglom":
        agglom = cl.AgglomerativeClustering(n_clusters=n_clusters).fit(X_scal)
        labels = agglom.labels_

    if cluster_type == "birch":
        birch = cl.Birch(n_clusters=n_clusters, threshold=0.2).fit(X_scal)
        labels = birch.labels_

    count = 0
    for label in range(n_clusters):
        label_msk = labels == label

        # Get group ID counts for accreted particles in this cluster
        grp_lst_acc = np.abs(rnd_grps[label_msk])
        unique_acc, counts_acc = np.unique(grp_lst_acc, return_counts=True)
        count_dict_acc = dict(zip(unique_acc, counts_acc))

        # Identify the most common group
        grp_identified = unique_acc[np.argmax(counts_acc)]

        if count_dict_acc[grp_identified] > count:
            count = count_dict_acc[grp_identified]
            grp_label = label
            pred_grp = grp_identified

    return pred_grp, grp_label, labels

In [3]:
def clustering_envelope(it, snap, proc_data, n_clusters, variables, cluster_type):
    it_id = gc_utils.iteration_name(it)
    snap_id = gc_utils.snapshot_name(snap)

    snp_dat = proc_data[it_id]["snapshots"][snap_id]
    acc_msk = snp_dat["now_accreted"][()] == 1
    bnd_msk = snp_dat["bound_flag"][()] == 1

    full_grps = np.abs(snp_dat["group_id"][acc_msk & bnd_msk])
    full_gcid = snp_dat["gc_id"][acc_msk & bnd_msk]

    rnd_grps = full_grps.copy()
    rnd_gcid = full_gcid.copy()

    unq_grp, cnt_grp = np.unique(full_grps, return_counts=True)
    grp_cnt_dict = dict(zip(unq_grp, cnt_grp))

    # res_dict = {
    #     gc_id: {"True": grp, "Pred": None} if grp in grp_lst else {"True": -1, "Pred": None}
    #     for gc_id, grp in zip(full_gcid, full_grps)
    # }

    # res_dict = {gc_id: {"True": grp, "Pred": None} for gc_id, grp in zip(full_gcid, full_grps)}

    res_dict = {
        gc_id: {"True": grp, "Pred": None} if grp_cnt_dict[grp] >= 5 else {"True": -1, "Pred": None}
        for gc_id, grp in zip(full_gcid, full_grps)
    }

    X_init = np.column_stack(
        [
            snp_dat["j.cyl"][:, 2][acc_msk & bnd_msk]
            if var == "jz"
            else snp_dat["j.cyl"][:, 1][acc_msk & bnd_msk]
            if var == "jp"
            else snp_dat["j.cyl"][:, 0][acc_msk & bnd_msk]
            if var == "jr"
            else snp_dat[var][acc_msk & bnd_msk]
            for var in variables
        ]
    )

    X = X_init

    while len(X) >= 5:
        pred_grp, grp_label, labels = clustering(X, rnd_grps, n_clusters, cluster_type)

        lab_msk = labels == grp_label
        gc_filt = rnd_gcid[lab_msk]

        for gc_id in gc_filt:
            res_dict[gc_id]["Pred"] = pred_grp

        rnd_grps = rnd_grps[~lab_msk]
        rnd_gcid = rnd_gcid[~lab_msk]
        X = X[~lab_msk]

    # group remaining gc's as other (-1)
    for gc_id in rnd_gcid:
        res_dict[gc_id]["Pred"] = -1

    return res_dict

In [4]:
def calc_entropy(p):
    """Compute Shannon entropy of a probability distribution p."""
    p = p[p > 0]  # filter out zeros to avoid log(0)
    return -np.sum(p * np.log2(p))


def clustering_entropy(true_labels, predicted_labels):
    labels = np.unique(np.concatenate((np.array(true_labels), np.array(predicted_labels))))
    cm = confusion_matrix(true_labels, predicted_labels, labels=labels)
    total = cm.sum()

    weighted_entropies = [] = []
    entropy_by_cluster = {}

    for i, col in enumerate(cm.T):  # loop over predicted clusters
        cluster_label = labels[i]
        cluster_total = np.sum(col)

        if cluster_total == 0:
            entropy_by_cluster[cluster_label] = 0.0
            weighted_entropies.append(0.0)
            continue

        p = col / cluster_total
        e = calc_entropy(p)
        weight = cluster_total / total

        entropy_by_cluster[cluster_label] = e / np.log2(cm.shape[1])
        weighted_entropies.append(weight * e)

    total_entropy = np.sum(weighted_entropies)

    # print(entropy_by_cluster)
    return total_entropy

In [5]:
# High F1 -> clear kinematic signature of a group (well-recovered).
# Low F1 -> the group is either mixed, fragmented, or indistinct.


def get_classifications(true_grp, pred_grp):
    report = classification_report(true_grp, pred_grp, output_dict=True, zero_division=0)

    class_dict = {}
    class_dict["macro_reca"] = report["macro avg"]["recall"]
    class_dict["macro_prec"] = report["macro avg"]["precision"]
    class_dict["macro_f1"] = report["macro avg"]["f1-score"]

    return report

In [6]:
def snapshot_clustering(it, proc_data, cluster_type, n_clusters, variables, snap_lst):
    snap_dict = {}
    snap_dict["entropy"] = []
    snap_dict["accuracy"] = []
    snap_dict["weight_recall"] = []
    snap_dict["macro_recall"] = []
    snap_dict["weight_precision"] = []
    snap_dict["macro_precision"] = []
    snap_dict["weight_f1"] = []
    snap_dict["macro_f1"] = []

    for snap in snap_lst:
        res_dict = clustering_envelope(it, snap, proc_data, n_clusters, variables, cluster_type)

        true_grp = [res_dict[gc_id]["True"] for gc_id in res_dict.keys()]
        pred_grp = [res_dict[gc_id]["Pred"] for gc_id in res_dict.keys()]

        entropy = clustering_entropy(true_grp, pred_grp)
        num_grps = len(np.unique(np.concatenate((true_grp, pred_grp))))
        entropy_norm = entropy / np.log2(num_grps)
        snap_dict["entropy"].append(entropy_norm)

        report = classification_report(true_grp, pred_grp, output_dict=True, zero_division=0)
        snap_dict["accuracy"].append(report["accuracy"])
        snap_dict["weight_recall"].append(report["weighted avg"]["recall"])
        snap_dict["macro_recall"].append(report["macro avg"]["recall"])
        snap_dict["weight_precision"].append(report["weighted avg"]["precision"])
        snap_dict["macro_precision"].append(report["macro avg"]["precision"])
        snap_dict["weight_f1"].append(report["weighted avg"]["f1-score"])
        snap_dict["macro_f1"].append(report["macro avg"]["f1-score"])

    return snap_dict

In [7]:
def iteration_clustering(it_lst, cluster_type, n_clusters, variables, sim, sim_dir, snap_lim):
    proc_file = sim_dir + sim + "/" + sim + "_processed.hdf5"
    proc_data = h5py.File(proc_file, "r")  # open processed data file

    pub_data = sim_dir + "snapshot_times_public.txt"
    pub_snaps = pd.read_table(pub_data, comment="#", header=None, sep=r"\s+")
    pub_snaps.columns = [
        "index",
        "scale_factor",
        "redshift",
        "time_Gyr",
        "lookback_time_Gyr",
        "time_width_Myr",
    ]

    snap_lst = pub_snaps["index"].values
    snap_lst = snap_lst[snap_lst >= snap_lim]
    time_lst = [pub_snaps[pub_snaps["index"] == snap]["time_Gyr"].values[0] for snap in snap_lst]

    # ent_it_lst = []
    it_dict = defaultdict(list)

    for it in it_lst:
        snap_dict = snapshot_clustering(it, proc_data, cluster_type, n_clusters, variables, snap_lst)
        for metric in snap_dict.keys():
            it_dict[metric].append(snap_dict[metric])

    cluster_dict = {
        "snap": snap_lst,
        "time": time_lst,
    }

    # Add mean and std of each metric across iterations
    for metric in it_dict.keys():
        cluster_dict[f"{metric}_avg"] = np.nanmean(it_dict[metric], axis=0)
        cluster_dict[f"{metric}_std"] = np.nanstd(it_dict[metric], axis=0)

    return cluster_dict

In [8]:
def averaged_metric(report, group_keys, metric="f1-score", exclude_group="0"):
    total_w_sum = 0.0
    total_m_sum = 0.0
    total_counts = 0

    num_grps = len(group_keys) - 1

    for group in group_keys:
        if group == exclude_group:
            continue

        value = report[group][metric]
        count = report[group]["support"]

        total_w_sum += value * count
        total_m_sum += value
        total_counts += count

    if total_counts == 0:
        w_avg = np.nan
        m_avg = np.nan
    else:
        w_avg = total_w_sum / total_counts
        m_avg = total_m_sum / num_grps

    return w_avg, m_avg

In [9]:
def group_snap_clustering(it, proc_data, cluster_type, n_clusters, variables, snap_lst):
    group_dict = {}
    for snap in snap_lst:
        res_dict = clustering_envelope(it, snap, proc_data, n_clusters, variables, cluster_type)

        true_grp = [res_dict[gc_id]["True"] for gc_id in res_dict.keys()]
        pred_grp = [res_dict[gc_id]["Pred"] for gc_id in res_dict.keys()]

        report = classification_report(true_grp, pred_grp, output_dict=True, zero_division=0)

        group_keys = [k for k in report.keys() if not any(c.isalpha() for c in str(k))]
        for group in group_keys:
            if group not in group_dict.keys():
                group_dict[group] = {"snap": [], "recall": [], "precision": [], "f1": []}

            group_dict[group]["snap"].append(snap)
            group_dict[group]["recall"].append(report[group]["recall"])
            group_dict[group]["precision"].append(report[group]["precision"])
            group_dict[group]["f1"].append(report[group]["f1-score"])

        if "ex_situ" not in group_dict.keys():
            group_dict["ex_situ"] = {
                "snap": [],
                "weight_recall": [],
                "macro_recall": [],
                "weight_precision": [],
                "macro_precision": [],
                "weight_f1": [],
                "macro_f1": [],
            }

        w_r, m_r = averaged_metric(report, group_keys, metric="recall")
        w_p, m_p = averaged_metric(report, group_keys, metric="precision")
        w_f, m_f = averaged_metric(report, group_keys, metric="f1-score")

        group_dict["ex_situ"]["snap"].append(snap)
        group_dict["ex_situ"]["weight_recall"].append(w_r)
        group_dict["ex_situ"]["macro_recall"].append(m_r)
        group_dict["ex_situ"]["weight_precision"].append(w_p)
        group_dict["ex_situ"]["macro_precision"].append(m_p)
        group_dict["ex_situ"]["weight_f1"].append(w_f)
        group_dict["ex_situ"]["macro_f1"].append(m_f)

    return group_dict

In [10]:
def group_iteration_clustering(it_lst, cluster_type, n_clusters, variables, sim, sim_dir, snap_lim):
    proc_file = sim_dir + sim + "/" + sim + "_processed.hdf5"
    proc_data = h5py.File(proc_file, "r")  # open processed data file

    pub_data = sim_dir + "snapshot_times_public.txt"
    pub_snaps = pd.read_table(pub_data, comment="#", header=None, sep=r"\s+")
    pub_snaps.columns = [
        "index",
        "scale_factor",
        "redshift",
        "time_Gyr",
        "lookback_time_Gyr",
        "time_width_Myr",
    ]

    snap_lst = pub_snaps["index"].values
    snap_lst = snap_lst[snap_lst >= snap_lim]
    time_lst = [pub_snaps[pub_snaps["index"] == snap]["time_Gyr"].values[0] for snap in snap_lst]

    # ent_it_lst = []
    # it_dict = defaultdict(list)
    it_dict = {}

    for it in it_lst:
        grp_snap_dict = group_snap_clustering(it, proc_data, cluster_type, n_clusters, variables, snap_lst)
        for key in grp_snap_dict.keys():
            if key not in it_dict:
                it_dict[key] = defaultdict(list)

            for metric in grp_snap_dict[key]:
                metric_lst = []
                for snap in snap_lst:
                    grp_snp_lst = np.array(grp_snap_dict[key]["snap"])
                    if snap not in grp_snp_lst:
                        metric_lst.append(np.nan)
                    else:
                        snp_msk = grp_snp_lst == snap
                        val = np.array(grp_snap_dict[key][metric])[snp_msk][0]
                        metric_lst.append(val)

                it_dict[key][metric].append(metric_lst)

    cluster_dict = {"snap": snap_lst, "time": time_lst, "groups": {}}

    # Add mean and std of each metric across iterations
    for key in it_dict.keys():
        if key == "ex_situ":
            cluster_dict[key] = {}
        else:
            cluster_dict["groups"][key] = {}
        for metric in it_dict[key].keys():
            if metric != "snap":
                if key == "ex_situ":
                    cluster_dict[key][f"{metric}_avg"] = np.nanmean(it_dict[key][metric], axis=0)
                    cluster_dict[key][f"{metric}_std"] = np.nanstd(it_dict[key][metric], axis=0)
                else:
                    cluster_dict["groups"][key][f"{metric}_avg"] = np.nanmean(it_dict[key][metric], axis=0)
                    cluster_dict["groups"][key][f"{metric}_std"] = np.nanstd(it_dict[key][metric], axis=0)

    cluster_dict["in_situ"] = {}
    for metric in cluster_dict["groups"]["0"].keys():
        cluster_dict["in_situ"][metric] = cluster_dict["groups"]["0"][metric]

    return cluster_dict

In [11]:
# about 2 minutes to run on 101 iterations
it_min = 0
it_max = 100

snap_lim = 600  # m12i: 60, m12f:
cluster_type = "agglom"  # birch, kmeans, agglom

variables = ["et_norm", "jr", "jp", "jz"]
n_clusters = 2

# snap_lim = 46
sim_dir = "/Users/z5114326/Documents/simulations/"
# sim_dir = "/Volumes/One_Touch/simulations/"
# sim_dir = "/Volumes/Expansion/simulations/"

# fire_dir = sim_dir + sim + "/" + sim + "_res7100"

pub_data = sim_dir + "snapshot_times_public.txt"
pub_snaps = pd.read_table(pub_data, comment="#", header=None, sep=r"\s+")
pub_snaps.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
snap_lst = pub_snaps["index"].to_numpy()
time_lst = pub_snaps["time_Gyr"].to_numpy()

snap_lst = pub_snaps[pub_snaps["index"] >= snap_lim]["index"].to_numpy()
time_lst = pub_snaps[pub_snaps["index"] >= snap_lim]["time_Gyr"].to_numpy()

sim = "m12i"
all_data = sim_dir + "/" + sim + "/" + sim + "_res7100/" + "snapshot_times.txt"
all_snaps = pd.read_table(all_data, comment="#", header=None, sep=r"\s+")
all_snaps.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]

it_csv = sim_dir + "iteration_check.csv"
it_df = pd.read_csv(it_csv)

In [12]:
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]
# sim_lst = ["m12i"]
# it_lst = np.arange(it_min, it_max + 1)

# for 101 iterations takes about 1 minute
# clsr_dict = iteration_clustering(it_lst, cluster_type, n_clusters, variables, sim, sim_dir, snap_lim)

# for 101 iterations takes about 1 minute
sim_clsr_dct = {}
for sim in sim_lst:
    it_msk = it_df[sim] == 0
    it_lst = np.array([int(it_id[2:]) for it_id in it_df["it_id"][it_msk].values])
    clsr_dct = group_iteration_clustering(it_lst, cluster_type, n_clusters, variables, sim, sim_dir, 600)
    sim_clsr_dct[sim] = clsr_dct

In [16]:
prop = "f1"  # recall, precision, f1

for sim in sim_lst:
    insitu_avg = np.round(sim_clsr_dct[sim]["in_situ"][prop + "_avg"][0], 2)
    insitu_std = np.round(sim_clsr_dct[sim]["in_situ"][prop + "_std"][0], 2)
    exsitu_avg = np.round(sim_clsr_dct[sim]["ex_situ"]["weight_" + prop + "_avg"][0], 2)
    exsitu_std = np.round(sim_clsr_dct[sim]["ex_situ"]["weight_" + prop + "_std"][0], 2)

    print(sim + ":", "in-situ", insitu_avg, "+/-", insitu_std, ", ", "ex-situ", exsitu_avg, "+/-", exsitu_std)

m12b: in-situ 0.51 +/- 0.25 ,  ex-situ 0.37 +/- 0.16
m12c: in-situ 0.86 +/- 0.06 ,  ex-situ 0.07 +/- 0.09
m12f: in-situ 0.78 +/- 0.12 ,  ex-situ 0.25 +/- 0.12
m12i: in-situ 0.84 +/- 0.06 ,  ex-situ 0.33 +/- 0.16
m12m: in-situ 0.82 +/- 0.06 ,  ex-situ 0.16 +/- 0.08


In [32]:
# BIRCH
# m12b: in-situ 0.43 +/- 0.3 ,  ex-situ 0.33 +/- 0.14
# m12c: in-situ 0.86 +/- 0.06 ,  ex-situ 0.09 +/- 0.09
# m12f: in-situ 0.75 +/- 0.13 ,  ex-situ 0.18 +/- 0.09
# m12i: in-situ 0.78 +/- 0.07 ,  ex-situ 0.17 +/- 0.11
# m12m: in-situ 0.81 +/- 0.1 ,  ex-situ 0.11 +/- 0.1

In [33]:
# kmeans
# m12b: in-situ 0.47 +/- 0.27 ,  ex-situ 0.36 +/- 0.12
# m12c: in-situ 0.86 +/- 0.06 ,  ex-situ 0.1 +/- 0.1
# m12f: in-situ 0.77 +/- 0.14 ,  ex-situ 0.27 +/- 0.1
# m12i: in-situ 0.82 +/- 0.08 ,  ex-situ 0.24 +/- 0.14
# m12m: in-situ 0.82 +/- 0.1 ,  ex-situ 0.15 +/- 0.1

In [34]:
# agglom BEST PERFORMING
# m12b: in-situ 0.51 +/- 0.25 ,  ex-situ 0.37 +/- 0.15
# m12c: in-situ 0.86 +/- 0.06 ,  ex-situ 0.07 +/- 0.09
# m12f: in-situ 0.78 +/- 0.12 ,  ex-situ 0.25 +/- 0.12
# m12i: in-situ 0.84 +/- 0.06 ,  ex-situ 0.32 +/- 0.17
# m12m: in-situ 0.82 +/- 0.1 ,  ex-situ 0.16 +/- 0.1

In [87]:
# for 101 iterations takes about 1 minute
sim = "m12c"
clsr_dict = iteration_clustering(it_lst, cluster_type, n_clusters, variables, sim, sim_dir, 102)

In [90]:
kin_file = sim_dir + sim + "/" + "galaxy_kinematics.json"
with open(kin_file, "r") as file:
    kin_dict = json.load(file)

pub_data = sim_dir + "snapshot_times_public.txt"
pub_snaps = pd.read_table(pub_data, comment="#", header=None, sep=r"\s+")
pub_snaps.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]

snap_lst = pub_snaps["index"].values
snap_lst = snap_lst[snap_lst >= 102]

kappa_co = []
for snap in snap_lst:
    snap_id = gc_utils.snapshot_name(snap)
    kap = kin_dict[snap_id]["kappa_co_sg"]
    kappa_co.append(kap)

In [92]:
scipy.stats.pearsonr(kappa_co, clsr_dict["weight_f1_avg"])
scipy.stats.pearsonr(kappa_co, clsr_dict["entropy_avg"])

PearsonRResult(statistic=0.5853281578827764, pvalue=0.0016824846233226365)

In [ ]:
# m12b: f1: statistic=-0.7693870673171994, pvalue=4.352722052277719e-06
# m12c: f1: statistic=-0.4713033145871069, pvalue=0.01508113014964238
# m12f: f1: statistic=-0.8649733086795193, pvalue=1.1966080421159591e-08
# m12i: f1: statistic=-0.9476664718318392, pvalue=2.1296798937032506e-13
# m12m: f1: statistic=-0.46315499609069904, pvalue=0.017180884490332057

In [ ]:
# m12b: entropy: statistic=0.8164179553728099, pvalue=3.6638290309173263e-07
# m12c: entropy: statistic=0.5853281578827764, pvalue=0.0016824846233226365
# m12f: entropy: statistic=0.8767081368003039, pvalue=4.280842078266133e-09
# m12i: entropy: statistic=0.9495290667846515, pvalue=1.392035963073304e-13
# m12m: entropy: statistic=0.14084085267479124, pvalue=0.4925467879531138